# 04 · Retrieval upgrade (lever 3)

**What changed (v2 to v3):** `retrieval.sources_for(agent)` restricts each specialist to its own source family, the query is rewritten with the life stage, and `search_reranked(query, source_ids, top_k=6, candidates=24)` reranks 24 candidates to 6.

**Hypothesis (plan §9.5 #3):** faithfulness +8 to 10 points.

In [1]:
from evals import compare

table = compare.comparison_table()
BEFORE, AFTER = "v2", "v3"
rows = [r for r in table if r["version"] in (BEFORE, AFTER)]
print(compare.format_table(rows))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
v2           100.0% (+0.0%)    99.0% (+0.0%) X    84.6% (+4.8%) X      80.8% (-1.4%)    79.4% (-6.3%) X     8.31 (+0.73) X      0.917 (-0.09)     1.541 (+0.028)
v3           100.0% (+0.0%)    99.0% (+0.0%) X    82.7% (-1.9%) X      78.1% (-2.7%)    76.1% (-3.3%) X    10.55 (+2.24) X     0.951 (+0.034)     1.555 (+0.014)
X = misses the §9.1 pass bar (citation match has no bar of its own); deltas are vs the previous row


## Metrics before and after

Saved results only: `evals/results/*_summary.json` plus the Nebius judge CSVs. Deltas are AFTER minus BEFORE; X marks a missed §9.1 pass bar.

In [2]:
print("faithfulness regressions v2 -> v3:", [r["case_id"] for r in compare.faithfulness_regressions("v2", "v3")])
print("unsupported-claim cases, v2:", len(compare.failure_modes("v2")["unsupported claim"]), " v3:", len(compare.failure_modes("v3")["unsupported claim"]))

faithfulness regressions v2 -> v3: ['B09', 'F07', 'F20', 'M02', 'M07', 'M09']
unsupported-claim cases, v2: 14  v3: 16


## Reading the result

- **The hypothesis was not met:** judged faithfulness fell 79.4% to 76.1% (-3.3 points, against a predicted +8 to 10), and citation match fell 80.8% to 78.1%.
- **Latency regressed:** p95 8.31 s to 10.55 s (the rerank hop). Routing 84.6% to 82.7%.
- Mean cost is flat (0.917 to 0.951 INR).
- Better passages alone did not stop unsupported claims: the model still adds statements the passages do not carry. That points at verification (notebook 05), not more retrieval tuning.

## Live run (off)

Re-running costs money (see the cost column) and writes a new LangSmith experiment. Saved results above are what this notebook reports.

In [3]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
    from nestling import graph

    graph.run_experiment("v3")
else:
    print("RUN_LIVE = False: showing saved results only")

RUN_LIVE = False: showing saved results only
